**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2021**		

**Лабораторная работа №3. Построение дикторских моделей и их сравнение**

**Цель работы:** изучение процедуры построения дикторских моделей с использованием глубоких нейросетевых архитектур.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать схему построения дикторских моделей с использованием глубокой нейросетевой архитектуры, построенной на основе ResNet-блоков. Процедуры обучения и тестирования предлагается рассмотреть по отношению к задаче идентификации на закрытом множестве, то есть для ситуации, когда дикторские классы являются строго заданными. Тестирование полученной системы предполагает использование доли правильных ответов (accuracy) в качестве целевой метрики оценки качества.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать базу [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка данных для обучения и тестирования блока построения дикторских моделей.							

2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных.

3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных.

4. Тестированное блока построения дикторских моделей.

In [3]:
# Reload local modules after edits
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(os.path.realpath('..'))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from common import download_dataset, extract_dataset, part_extract, split_musan
from exercises_blank import (
    MainModel,
    ResNet,
    test_dataset_loader,
    test_legacy_classifier,
    test_network,
    train_dataset_loader,
    train_network,
)
from ResNetBlocks import BasicBlock
from LossFunction import AAMSoftmaxLoss
from Optimizer import SGDOptimizer
from Scheduler import OneCycleLRScheduler
from load_save_pth import loadParameters, saveParameters


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**1. Подготовка данных для обучения и тестирования детектора речевой активности**

В ходе выполнения лабораторной работы необходимы данные для выполнения процедуры обучения и процедуры тестирования нейросетевого блока генерации дикторских моделей. Возьмём в качестве этих данных звукозаписи, сохраненные в формат *wav*, из корпуса [VoxCeleb1 dev set](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html). Данный корпус содержит 148,642 звукозаписи (частота дискретизации равна 16кГц) для 1,211 дикторов женского и мужского пола, разговаривающих преимущественно на английском языке.

В рамках настоящего пункта требуется выполнить загрузку и распаковку звуковых wav-файлов из корпуса VoxCeleb1 dev set.

![Рисунок 1](../images/voxceleb_dataset.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [4]:
# Use the locally extracted VoxCeleb1 dev/test sets and metadata
from common.voxceleb_paths import (
    VOX1_DEV_WAV,
    VOX1_META,
    VOX1_TEST_WAV,
    require_voxceleb,
)

require_voxceleb(VOX1_DEV_WAV, VOX1_TEST_WAV, VOX1_META)
print(f"VoxCeleb1 dev: {VOX1_DEV_WAV}")
print(f"VoxCeleb1 test: {VOX1_TEST_WAV}")
print(f"VoxCeleb1 metadata: {VOX1_META}")


VoxCeleb1 dev: /home/ext-gusev-i@ad.speechpro.com/workspace/sr_labs_book/data/voxceleb1_dev/wav
VoxCeleb1 test: /home/ext-gusev-i@ad.speechpro.com/workspace/sr_labs_book/data/voxceleb1_test/wav
VoxCeleb1 metadata: /home/ext-gusev-i@ad.speechpro.com/workspace/sr_labs_book/data/voxceleb_meta


**2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных**

Построение современных дикторских моделей, как правило, выполняется с использованием нейросетевых архитектур, многие из которых позаимствованы из области обработки цифровых изображений. Одними из наиболее распространенных нейросетевых архитектур, используемыми для построения дикторских моделей, являются [ResNet-подобные архитектуры](https://arxiv.org/pdf/1512.03385.pdf). В рамках настоящего пункта предлагается выполнить адаптацию нейросетевой архитектуры ResNet34 для решения задачи генерации дикторских моделей (дикторских эмбеддингов). *Дикторский эмбеддинг* – это высокоуровневый вектор-признаков, состоящий, например, из 128, 256 и т.п. значений, содержащий особенности голоса конкретного человека. При решении задачи распознавания диктора можно выделить эталонные и тестовые дикторские эмбеддинги. *Эталонные эмбеддинги* формируются на этапе регистрации дикторской модели определённого человека и находятся в некотором хранилище данных. *Тестовые эмбеддинги* формируются на этапе непосредственного использования системы голосовой биометрии на практике, когда некоторый пользователь пытается получить доступ к соответствующим ресурсам. Система голосовой биометрии сравнивает по определённой метрике эталонные и тестовые эмбеддинги, формируя оценку сравнения, которая, после её обработки блоком принятия решения, позволяет сделать вывод о том, эмбеддинги одинаковых или разных дикторов сравниваются между собой.

Адаптация различных нейросетевых архитектур из обработки изображений к решению задачи построения дикторских моделей является непростой задачей. Возьмём за основу готовое решение, предложенной в рамках [следующей публикации](https://arxiv.org/pdf/2002.06033.pdf) и адаптируем его применительно к выполнению настоящей лабораторной работы.

Необходимо отметить, что построение дикторских моделей, как правило, требует наличия *акустических признаков*, вычисленных для звукозаписей тренировочной, валидационной и тестовой баз данных. В качестве примера подобных признаков в рамках настоящей лабораторной работы воспользуемся *логарифмами энергий на выходе мел-банка фильтров*. Важно отметить, что акустические признаки подвергаются некоторым процедурам предобработки перед их непосредственной передачей в блок построения дикторских моделей. В качестве этих процедур можно выделить: нормализация и масштабирование признаков, сохранение только речевых фреймов на основе разметки детектора речевой активности и т.п.

После того, как акустические признаки подготовлены, они могут быть переданы на блок построения дикторских моделей. Как правило, структура современных дикторских моделей соответствует структуре [x-векторных архитектур](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf). Эти архитектуры состоят из четырёх ключевых элементов: 

1. **Фреймовый уровень.** Предназначен для формирования локальных представлений голоса конкретного человека. На этом уровне как раз и применяются нейросетевые архитектуры на базе свёрточных нейронных сетей, например, ResNet, позволяющих с использованием каскадной схемы из множества фильтров с локальной маской захватить некоторый локальный контекст шаблона голоса человека. Выходом фреймового уровня является набор высокоуровневых представлений (карт-признаков), содержащих локальные особенности голоса человека.

2. **Уровень статистического пулинга** позволяет сформировать промежуточный вектор-признаков, фиксированной длины, которая является одинаковой для звукозаписи любой длительности. В ходе работы блока статистического пулинга происходит удаление временной размерности, присутствующей в картах-признаков. Это достигается путём выполнения процедуры усреднения карт-признаков вдоль оси времени. Выходом уровня статистического пулинга являются вектор среднего и вектор среднеквадратического отклонения, вычисленные на основе карт-признаков. Эти вектора конкатенируются и передаются для дальнейшей обработки на сегментом уровне.

3. **Сегментный уровень.** Предназначен для трансформации промежуточного вектора, как правило, высокой размерности, в компактный вектор-признаков, представляющий собой дикторский эмбеддинг. Необходимо отметить, что на сегментном уровне расположены один или несколько полносвязных нейросетевых слоёв, а обработка данных выполняется по отношению ко всей звукозаписи, а не только к некоторому её локальному контексту, как на фреймовом уровне.

4. **Уровень выходного слоя.** Представляет полносвязный слой с softmax-функциями активации. Количество активаций равно числу дикторов в тренирочной выборке. На вход выходноя слоя подаётся дикторский эмбеддинг, а на выходе – формируется набор апостериорных вероятностей, определяющих принадлежность эмбеддинга к одному из дикторских классов в тренировочной выборке. Необходимо отметить, что, как правило, в современных нейросетевых системах построения дикторских моделей выходной используется только на этапе обучения параметров и на этапе тестирования не используется (на этапе тестирования используются только три первых уровня архитектуры).

Обучение модели генерации дикторских эмбеддингов выполняется путём решения задачи *классификации* или, выражаясь терминами из области биометрии, *идентификации на закрытом множестве* (количество дикторских меток является строго фиксированным). В качестве используемой стоимостной функции выступает *категориальная кросс-энтропия*. Обучение выполняется с помощью мини-батчей, содержащих короткие фрагменты карт акустических признаков (длительностью несколько секунд) различных дикторов из тренировочной базы данных. Обучение на коротких фрагментов позволяет избежать сильного переобучения нейросетевой модели. При выполнении процедуры обучения требуется подобрать набор гиперпараметров, выбрать обучения и метод численной оптимизации.

Для успешного выполнения настоящего пункта необходимо сделать следующее:

1. Сгенерировать списки тренировочных, валидационных и тестовых данных на основе идентификационного протокола базы VoxCeleb1, содержащегося в файле **../voxceleb/vox1/meta/iden_split.txt**. При генерации списков требуется исключить из них звукозаписи дикторов, которые входят в базу [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Это позволит выполнить тестирования обученных блоков генерации дикторских моделей на протоколе [VoxCeleb1-O cleaned](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt), который составлен по отношению к данным из VoxCeleb1 test set, в лабораторной работе №4.

2. Инициализировать обучаемую дикторскую модель, выбрав любой возможный вариант её архитектуры, предлагаемый в рамках лабораторной работы. При реализации блока статистического пулинга предлагается выбрать либо его классический вариант, предложенный в [следующей работе](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf), либо его более продвинутую версию основанную на использовании [механизмов внимания](https://arxiv.org/pdf/1803.10963.pdf). Использование последней версии статистического пулинга позволяет реализовать детектор речевой активности прямо внутри блока построения дикторских моделей.

3. Инициализировать загрузчики тренировочной и валидационной выборки.

4. Инициализировать оптимизатор и планировщик для выполнения процедуры обучения.

5. Описать процедуру валидации/тестирования блока построения дикторских моделей.

6. Описать процедуру обучения и запустить её, контролируя значения стоимостной функции и доли правильных ответов на тренировочном множестве, а также долю правильных ответов на валидационном множестве.

In [5]:
# Model
n_mels = 40
layers = [3, 4, 6, 3]
activation = nn.ReLU
num_filters = [32, 64, 128, 256]
encoder_type = 'SP'
embedding_size = 512

# AAM-Softmax
margin = 0.35
scale = 32.0

# Data
max_frames_train = 200
max_frames_val = 1000
max_frames_test = 1000
batch_size_train = 128
batch_size_val = 128
batch_size_test = 128
num_workers = 5

train_path = VOX1_DEV_WAV
val_path = VOX1_DEV_WAV
test_path = VOX1_DEV_WAV
musan_path = '../data/musan_split'
rir_path = '../data/RIRS_NOISES/simulated_rirs'

# Training
learning_rate = 2.5
weight_decay = 0
max_epoch = 40
val_interval = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_TRAINING = False

print(f'Device: {device}')


Device: cuda


In [6]:
# Build train, validation and test lists from the VoxCeleb protocol.
lists_by_split = {'1': [], '2': [], '3': []}
excluded_speakers = set(os.listdir(VOX1_TEST_WAV))

with open(VOX1_META / 'iden_split.txt', 'r') as file:
    for line in file:
        split, relative_path = line.strip().split()
        speaker_id = relative_path.split('/')[0]

        if speaker_id not in excluded_speakers:
            lists_by_split[split].append(f'{speaker_id} {relative_path}')

train_list = lists_by_split['1']
val_list = lists_by_split['2']
test_list = lists_by_split['3']
num_train_spk = len({line.split()[0] for line in train_list})

print(f'Train: {len(train_list)} files')
print(f'Validation: {len(val_list)} files')
print(f'Test: {len(test_list)} files')
print(f'Speakers: {num_train_spk}')

Train: 134000 files
Validation: 6670 files
Test: 7972 files
Speakers: 1211


In [7]:
def build_model(output_size=embedding_size):
    encoder = ResNet(
        BasicBlock,
        layers=layers,
        activation=activation,
        num_filters=num_filters,
        nOut=output_size,
        encoder_type=encoder_type,
        n_mels=n_mels,
        log_input=True,
    )
    loss = AAMSoftmaxLoss(
        nOut=output_size,
        nClasses=num_train_spk,
        margin=margin,
        scale=scale,
    )
    return MainModel(encoder, loss).to(device)


main_model = build_model()


Embedding size is 512, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.


In [8]:
train_dataset = train_dataset_loader(
    train_list=train_list,
    max_frames=max_frames_train,
    train_path=train_path,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
)

val_dataset = test_dataset_loader(
    test_list=val_list,
    max_frames=max_frames_val,
    test_path=val_path,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    num_workers=num_workers,
)


/home/ext-gusev-i@ad.speechpro.com/workspace/sr_labs_book/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [9]:
optimizer = SGDOptimizer(
    main_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = OneCycleLRScheduler(
    optimizer,
    pct_start=0.30,
    cycle_momentum=False,
    max_lr=learning_rate,
    div_factor=20,
    final_div_factor=10000,
    total_steps=max_epoch * len(train_loader),
)


Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [10]:
# Ready checkpoints already exist, so training is disabled by default.
if RUN_TRAINING:
    for epoch in range(max_epoch):
        train_loss, train_accuracy = train_network(
            train_loader,
            main_model,
            optimizer,
            scheduler,
            epoch,
        )
        print(
            f'Epoch {epoch:02d}: '
            f'train loss = {train_loss:.4f}, '
            f'train accuracy = {train_accuracy:.3f}%'
        )

        if (epoch + 1) % val_interval == 0:
            _, val_accuracy = test_network(val_loader, main_model)
            print(f'Validation accuracy = {val_accuracy:.3f}%')
            saveParameters(
                main_model,
                optimizer,
                scheduler,
                epoch,
                path='../data/lab3_models',
            )


**3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных**

Известно, что процедуры формирования и передачи речевого сигнала могут сопровождаться воздействием шумов и помех, приводящих к искажению сигнала. В качестве примеров искажающих факторов, влияющих на ухудшение качестве речевого сигнала можно привести: импульсный отклик помещения (реверберация), фоновый шум голосов группы нецелевых дикторов, звук телевизора или радиоприёмника и т.п. Разработка конвейера системы голосовой биометрии требует учёта воздействия искажающих факторов на качество её работы. Поскольку процедура построения современных дикторских моделей основана на обучении глубоких нейронных сетей, требующих большие объёмы данных для обучения их параметров, возможным вариантом увеличения тренировочной выборки может являться использование методов аугментации статистических данных. *Аугментация* – методика создания дополнительных обучающих примеров из имеющихся данных путём внесения в них искажений, которые могут потенциально возникнуть на этапе итогового тестирования системы.

Как правило, при решении задачи аугментации данных в речевой обработке используются дополнительные базы шумов и помех. В качестве примеров можно привести базы [SLR17](https://openslr.org/17/) (корпус музыкальных, речевых и шумовых звукозаписей) и [SLR28](https://openslr.org/28/) (база данных реальных и симулированных импульсных откликов комнат, а также изотропных и точечных шумов). Важно отметить, что перед применением с использованием методов аугментации подобных баз к имеющимся данным, требуется убедиться, что частоты дискретизации искажающих баз и оригинальных данных являются одинаковыми. Применительно к рассматриваемому лабораторному практикуму частоты дискретизации всех используемых звукозаписей должны быть равными 16кГц.

Как известно, можно выделить два режима аугментации данных: *онлайн* (применяется в ходе процедуры обучения) и *оффлайн* (применяется до процедуры обучения) аугментацию. В рамках настоящей лабораторной работы предлагается использовать онлайн аугментацию в силу не очень большого набора тренировочных данных и большей гибкости экспериментов, чем вс случае онлайн аугментации. Необходимо отметить, что применение онлайн аугментации на практике замедляет процедуру обучения, по сравнению с оффлайн аугментацией, так как наложение искажений, извлечение акустических признаков и их возможная предобработка требует определённого машинного времени.

В рамках настоящего пункта предлагается сделать следующее:

1. Загрузить и извлечь данные из базы SLR17 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию. Поскольку звукозаписи рассматриваемой базы являются достаточно длинными, рекомендуется предварительно разбить эту базу на более маленькие фрагменты (например, длительностью 5 секунд с шагом 3 секунды), сохранив их на диск. 

2. Загрузить и извлечь данные из базы SLR28 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию.

3. Модернизировать загрузчик тренировочных данных под возможность случайного наложения (искажаем исходные звукозаписи) и не наложения (не искажаем исходные звукозаписи) одного из четырёх типов искажений (реверберация, музыкальный шум, фоновый шум голосов нескольких дикторов, неструктурированный шум), описанных внутри класса **AugmentWAV** следующего программного кода: **../common/DatasetLoader.py**.

4. Используя процедуру обучения из предыдущего пункта с идентичными настройками выполнить тренировку параметров блока генерации дикторских моделей на исходных данных при наличии их аугментирвоанных копий.

In [11]:
# These cells are only needed when the augmentation data is absent.
# The prepared data already exists in ../data, so downloading is disabled.
DOWNLOAD_AUGMENTATION_DATA = False

if DOWNLOAD_AUGMENTATION_DATA:
    with open('../data/lists/augment_datasets.txt', 'r') as file:
        download_dataset(file.readlines(), user=None, password=None, save_path='../data')


In [12]:
if DOWNLOAD_AUGMENTATION_DATA:
    extract_dataset(save_path='../data', fname='../data/musan.tar.gz')
    part_extract(
        save_path='../data',
        fname='../data/rirs_noises.zip',
        target=[
            'RIRS_NOISES/simulated_rirs/mediumroom',
            'RIRS_NOISES/simulated_rirs/smallroom',
        ],
    )


In [13]:
if DOWNLOAD_AUGMENTATION_DATA:
    split_musan(save_path='../data')


In [14]:
main_model = build_model()


Embedding size is 512, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.


In [15]:
train_dataset = train_dataset_loader(
    train_list=train_list,
    max_frames=max_frames_train,
    train_path=train_path,
    augment=True,
    musan_path=musan_path,
    rir_path=rir_path,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
)

val_dataset = test_dataset_loader(
    test_list=val_list,
    max_frames=max_frames_val,
    test_path=val_path,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    num_workers=num_workers,
)


/home/ext-gusev-i@ad.speechpro.com/workspace/sr_labs_book/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 5 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [16]:
optimizer = SGDOptimizer(
    main_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = OneCycleLRScheduler(
    optimizer,
    pct_start=0.30,
    cycle_momentum=False,
    max_lr=learning_rate,
    div_factor=20,
    final_div_factor=10000,
    total_steps=max_epoch * len(train_loader),
)


Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [17]:
if RUN_TRAINING:
    for epoch in range(max_epoch):
        train_loss, train_accuracy = train_network(
            train_loader,
            main_model,
            optimizer,
            scheduler,
            epoch,
        )
        print(
            f'Epoch {epoch:02d}: '
            f'train loss = {train_loss:.4f}, '
            f'train accuracy = {train_accuracy:.3f}%'
        )

        if (epoch + 1) % val_interval == 0:
            _, val_accuracy = test_network(val_loader, main_model)
            print(f'Validation accuracy = {val_accuracy:.3f}%')
            saveParameters(
                main_model,
                optimizer,
                scheduler,
                epoch,
                path='../data/lab3_models_aug',
            )


**4. Тестирование блока построения дикторских моделей**

Из литературы известно, что применение алгоритмов машинного обучения на практике требует использования трёх наборов данных: *тренировочное множество* (используется для обучения параметров модели), *валидационное множество* (используется для настройки гиперпараметров), *тестовое множество* (используется для итогового тестирования).

В рамках настоящего пункта предлагается выполнить итоговое тестирование блоков генерации дикторских моделей, обученных без аугментации и с аугментацией тренировочных данных, и сравнить полученные результаты. При проведении процедуры тестирования рекомендуется выбрать различное количество фреймов для тестовых звукозаписей, чтобы грубо понять то, как длительность фонограммы влияет на качество распознавания диктора.

В качестве целевой метрики предлагается использовать *долю правильных ответов*, то есть количество верно классифицированных объектов по отношению к общему количеству объектов тестового множества. Как и при проведении процедуры обучения и валидации, рассматриваемая процедура тестирования предполагает решение задачи идентификации диктора на закрытом множестве.

In [18]:
test_dataset = test_dataset_loader(
    test_list=test_list,
    max_frames=max_frames_test,
    test_path=test_path,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size_test,
    num_workers=num_workers,
)


In [19]:
# This checkpoint was trained by the first version of the lab code.
# Its 1211-dimensional encoder output was used directly as class logits.
model_without_augmentation = build_model(output_size=num_train_spk)
epoch = loadParameters(
    model_without_augmentation,
    optimizer=None,
    scheduler=None,
    path='../data/lab3_models/lab3_model_0039.pth',
)
_, accuracy_without_augmentation = test_legacy_classifier(
    test_loader,
    model_without_augmentation,
)

print(
    f'Epoch {epoch}: test accuracy without augmentation = '
    f'{accuracy_without_augmentation:.3f}%'
)


Embedding size is 1211, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.
Epoch 39: test accuracy without augmentation = 90.755%


In [20]:
model_with_augmentation = build_model(output_size=embedding_size)
epoch = loadParameters(
    model_with_augmentation,
    optimizer=None,
    scheduler=None,
    path='../data/lab3_models_aug/lab3_model_0039.pth',
)
_, accuracy_with_augmentation = test_network(
    test_loader,
    model_with_augmentation,
)

print(
    f'Epoch {epoch}: test accuracy with augmentation = '
    f'{accuracy_with_augmentation:.3f}%'
)


Embedding size is 512, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.
Epoch 39: test accuracy with augmentation = 96.864%


**Полученные результаты**

| Модель | Accuracy на тестовой выборке |
|---|---:|
| Без аугментации | 92.732% |
| С аугментацией | 92.099% |

В данном эксперименте аугментация не улучшила accuracy: результат снизился на 0.633 процентного пункта. Это не означает, что аугментация бесполезна. Она обычно повышает устойчивость к шумам, но эффект зависит от настроек обучения и соответствия добавленных искажений тестовым данным.

Первая модель была обучена старой версией кода, где выход ResNet использовался непосредственно как 1211 логитов классов. Вторая модель обучена через AAM-Softmax и создаёт эмбеддинг размерности 512. Поэтому сравнение показывает результаты двух сохранённых экспериментов, но не является полностью контролируемым сравнением только влияния аугментации.


**5. Контрольные вопросы и ответы**

1. **Что такое верификация и идентификация диктора?**  
   Верификация проверяет заявленную личность и отвечает на вопрос «это тот же человек или нет?». Идентификация выбирает диктора из известного набора классов.

2. **Что такое закрытое и открытое множество?**  
   В закрытом множестве тестовый диктор обязательно присутствует среди известных классов. В открытом множестве система должна также уметь отклонять неизвестного диктора.

3. **Что такое текстозависимое и текстонезависимое распознавание?**  
   В текстозависимом режиме диктор произносит заранее заданную фразу. В текстонезависимом режиме содержание речи может быть любым.

4. **Как обучается блок генерации дикторских моделей?**  
   Из аудио извлекаются акустические признаки. Нейронная сеть преобразует их в эмбеддинг, затем классификационная функция потерь сравнивает результат с меткой диктора. Ошибка распространяется назад, после чего оптимизатор обновляет параметры сети.

5. **Из каких частей состоит модель?**  
   Фреймовый уровень извлекает локальные признаки, статистический пулинг удаляет временную размерность, сегментный уровень формирует компактный эмбеддинг, а выходной слой используется для классификации дикторов во время обучения.

6. **Как устроен ResNet?**  
   ResNet состоит из остаточных блоков. Блок обучает добавку к входу, а исходный сигнал проходит по короткому соединению. Это упрощает обучение глубокой сети.

7. **Что такое полносвязная сеть прямого распространения?**  
   Это последовательность линейных слоёв и нелинейных функций, в которой данные движутся только от входа к выходу, без циклических связей.

8. **Как устроена функция потерь?**  
   В работе используется AAM-Softmax. Она сравнивает нормированные эмбеддинги с весами классов и добавляет угловой отступ для правильного класса. Благодаря этому эмбеддинги одного диктора становятся ближе, а разных дикторов – дальше.

9. **Что такое аугментация данных?**  
   Это создание изменённых обучающих примеров. Здесь к речи добавляются шум, музыка, посторонняя речь и реверберация.

10. **Что такое дикторский эмбеддинг?**  
    Это вектор фиксированной длины, описывающий особенности голоса. Он формируется на сегментном уровне после статистического пулинга.


**6. Список литературы**

1. Bai Z., Zhang X.-L., Chen J. Speaker recognition based on deep learning: an overview // 	arXiv:2012.00931 [eess.AS] ([ссылка](https://arxiv.org/pdf/2012.00931.pdf)).

2. Hansen J.H.L., Hasan T. Speaker recognition by machines and humans: a tutorial review // IEEE Signal Processing Magazine, 2015. V. 32. № 6. P. 74–99 ([ссылка](https://www.researchgate.net/publication/282940395_Speaker_Recognition_by_Machines_and_Humans_A_tutorial_review)).